In [ ]:
import os
import numpy as np
from PIL import Image, ImageFilter
import openslide
from trident.Converter import AnyToTiffConverter

# 1) Your sweep definitions:
sweep_ranges = {
    "brightness":     (0.0, 30.0),
    "gaussian_noise": (0.0, 20.0),
    "gaussian_blur":  (0.0, 7.0),
    "jpeg":           (100, 90),
}
n_steps = {
    "brightness":     10,
    "gaussian_noise": 10,
    "gaussian_blur":  10,
    "jpeg":           10,
}
step_sizes = {
    name: (max_ - min_) / (n_steps[name] - 1)
    for name, (min_, max_) in sweep_ranges.items()
}

# 2) Custom perturbation functions:
def brightness_perturb(img: Image.Image, delta: float) -> Image.Image:
    import numpy as np
    arr = np.array(img, dtype=np.float32)
    arr += delta if np.random.rand() < 0.5 else -delta
    return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

def gaussian_noise_perturb(img: Image.Image, sigma: float) -> Image.Image:
    import numpy as np
    arr = np.array(img, dtype=np.float32)
    arr += np.random.normal(0, sigma, arr.shape)
    return Image.fromarray(arr.clip(0, 255).astype(np.uint8))

def gaussian_blur_perturb(img: Image.Image, sigma: float) -> Image.Image:
    return img.filter(ImageFilter.GaussianBlur(radius=sigma))

def jpeg_perturb(img: Image.Image, quality: float) -> Image.Image:
    from io import BytesIO
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=int(quality))
    buf.seek(0)
    return Image.open(buf).convert("RGB")

perturb_funcs = {
    "brightness":     brightness_perturb,
    "gaussian_noise": gaussian_noise_perturb,
    "gaussian_blur":  gaussian_blur_perturb,
    "jpeg":           jpeg_perturb,
}

# 3) Extraction helper
def extract_wsi_to_pil(wsi_path: str, target_mag: float):
    slide     = openslide.OpenSlide(wsi_path)
    base_mag  = float(slide.properties.get('openslide.objective-power', 20))
    mpp_base  = float(slide.properties.get('openslide.mpp-x', 0.5))
    mpp_target = mpp_base * (base_mag / target_mag)
    level     = slide.get_best_level_for_downsample(base_mag / target_mag)
    W, H      = slide.level_dimensions[level]
    img       = slide.read_region((0,0), level, (W,H)).convert("RGB")
    return img, mpp_target

# 4) Main pipeline
def run_wsi_custom_pipeline(
    wsi_dir: str,
    target_mag: float,
    out_root: str,
    perturb_funcs: dict,
    sweep_ranges: dict,
    step_sizes: dict,
    n_steps: dict,
    save_png: bool = False
):
    """
    save_png: if True, keeps PNG files alongside TIFFs; otherwise removes PNGs after conversion.
    """
    os.makedirs(out_root, exist_ok=True)

    # list all .tif files
    files = os.listdir(wsi_dir)
    tif_paths = [
        os.path.join(wsi_dir, f)
        for f in files
        if os.path.isfile(os.path.join(wsi_dir, f)) and f.lower().endswith(".tif")
    ]
    print(f"Found {len(tif_paths)} WSIs in {wsi_dir}")

    for wsi_path in tif_paths:
        base = os.path.splitext(os.path.basename(wsi_path))[0]
        print(f"\nProcessing WSI: {base}")

        # extract once
        img, mpp_target = extract_wsi_to_pil(wsi_path, target_mag)

        # for each perturbation type
        for name, fn in perturb_funcs.items():
            print(f" → {name}")
            start, end = sweep_ranges[name]
            step = step_sizes[name]
            # build raw list of values
            vals = [start + i*step for i in range(n_steps[name])]
            # if descending (e.g. JPEG)
            if end < start:
                vals = vals[::-1]
            # drop zero if present at start
            if start == 0.0 and vals and abs(vals[0]) < 1e-8:
                vals = vals[1:]

            # prepare output subdir for this perturbation
            pert_dir = os.path.join(out_root, name)
            os.makedirs(pert_dir, exist_ok=True)
            converter = AnyToTiffConverter(job_dir=pert_dir, bigtiff=True)

            # process each magnitude
            for mag in vals:
                pert_img = fn(img, mag)

                # save PNG
                png_name = f"{base}_{name}_{mag:.3f}.png"
                png_path = os.path.join(pert_dir, png_name)
                pert_img.save(png_path)

                # convert to TIFF (same folder)
                converter.process_file(png_path, mpp=mpp_target, zoom=1)

                # optionally remove PNG
                if not save_png:
                    os.remove(png_path)

    print("\n✅ Done all WSIs.")

# 5) Execute
wsi_dir  = os.path.join("..", "..", "..", "data", "CAMELYON", "test")
out_root = os.path.join(wsi_dir, "perturbed_images")

run_wsi_custom_pipeline(
    wsi_dir       = wsi_dir,
    target_mag    = 2.5,
    out_root      = out_root,
    perturb_funcs = perturb_funcs,
    sweep_ranges  = sweep_ranges,
    step_sizes    = step_sizes,
    n_steps       = n_steps,
    save_png      = False   # set True if you want to keep PNGs
)

In [ ]:
# import openslide

# def inspect_tiff(tif_path):
#     slide = openslide.OpenSlide(tif_path)
#     print(f"=== Inspecting {tif_path} ===")
#     print(f"OpenSlide backend: {slide.__class__.__name__}")
#     print(f"Number of levels (pyramid): {slide.level_count}")
#     for level in range(slide.level_count):
#         print(f" Level {level}: size={slide.level_dimensions[level]}, downsample={slide.level_downsamples[level]}")
#     print("\nProperties (metadata):")
#     for k, v in slide.properties.items():
#         print(f"  {k}: {v}")
#     print("\nAssociated images:")
#     for name in slide.associated_images:
#         print(f"  {name}")

#     # Specifically print MPP info
#     mpp_x = slide.properties.get('openslide.mpp-x')
#     mpp_y = slide.properties.get('openslide.mpp-y')
#     if mpp_x and mpp_y:
#         print(f"\nMPP x: {mpp_x} μm/px, y: {mpp_y} μm/px")

# # Example usage:
# tif_path = r"C:\Users\Dhyey\Documents\University\MSc\CS907 Dissertation Project\Tutorials\Data\CAMELYON\test\perturbed_images\brightness\test_001_brightness_0.000.tiff"
# inspect_tiff(tif_path)

In [16]:
def split_paths_by_size(paths: list[str], n_jobs: int) -> list[list[str]]:
    """
    Greedy “bin-packing” split of `paths` into `n_jobs` bins, balancing total file size.
    Returns a list of length n_jobs, each a list of paths.
    """
    # 1) get sizes
    ps = [(p, os.path.getsize(p)) for p in paths]
    # 2) sort descending by size
    ps.sort(key=lambda x: x[1], reverse=True)
    # 3) init bins
    bins = [[] for _ in range(n_jobs)]
    bin_sizes = [0]*n_jobs
    # 4) assign each file to the currently smallest bin
    for p, sz in ps:
        i = min(range(n_jobs), key=lambda j: bin_sizes[j])
        bins[i].append(p)
        bin_sizes[i] += sz
    return bins

In [17]:
import os
wsi_dir  = "../../../data/CAMELYON/test"
all_wsis = sorted(os.path.join(wsi_dir,f) 
                  for f in os.listdir(wsi_dir) if f.lower().endswith(".tif"))
bins     = split_paths_by_size(all_wsis, n_jobs=4)


In [19]:
for bin in bins:
    print(bin)

['data/CAMELYON/test\\test_006 - Copy.tif', 'data/CAMELYON/test\\test_008 - Copy.tif', 'data/CAMELYON/test\\test_001 - Copy.tif']
['data/CAMELYON/test\\test_006.tif', 'data/CAMELYON/test\\test_008.tif', 'data/CAMELYON/test\\test_001.tif']
['data/CAMELYON/test\\test_002 - Copy.tif', 'data/CAMELYON/test\\test_007 - Copy.tif', 'data/CAMELYON/test\\test_005 - Copy.tif']
['data/CAMELYON/test\\test_002.tif', 'data/CAMELYON/test\\test_007.tif', 'data/CAMELYON/test\\test_005.tif']
